# 1장 LLM은 어떻게 글을 만드는가 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers torch
# 1장의 예제는 PC에서 공개 소형 LLM을 실행해 토큰과 확률을 직접 봅니다. 처음 한 번 약 1GB를 내려받습니다
# API 키가 없어도 예제 1-1~1-3과 실습은 실행할 수 있습니다. 아래 연결 확인에서 오류가 나도
# "PC에서 실행하는 모델" 줄이 출력됐으면 계속 진행합니다
import embed
import torch
LOCAL_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(LOCAL_MODEL)          # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
transformers_logging.disable_progress_bar()
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL)
model.eval()                              # 학습하지 않고 계산만 합니다
print('PC에서 실행하는 모델:', LOCAL_MODEL)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 1-1. 토큰으로 나눠 보기

교재의 절: "토큰과 토크나이저"

같은 뜻의 한국어 문장과 영어 문장, 숫자가 든 문장을 토크나이저로 나눠 토큰 수를 비교합니다. 토큰을 하나씩 글로 되돌려 출력하므로 어디서 나뉘었는지 볼 수 있습니다. 깨진 글자(�)로 보이는 토큰은 한글 한 글자의 바이트 조각입니다.

In [ ]:
sentences = [
    '웨이퍼를 개별 칩으로 자르는 공정을 다이싱이라고 합니다.',
    'Dicing is the process of cutting a wafer into individual chips.',
    '와이어 본딩 온도 기준은 175°C입니다.',
]
print(f'어휘 크기: {len(tokenizer):,}개')
for sentence in sentences:
    token_ids = tokenizer.encode(sentence)
    pieces = [tokenizer.decode([token_id]) for token_id in token_ids]  # 토큰을 하나씩 글로 되돌립니다
    print(f'\n{len(sentence)}자, {len(token_ids)}토큰')
    print(' | '.join(pieces))

# 앞에 공백이 있는지에 따라 같은 글자도 토큰 수가 다릅니다
print('\n"칩"의 토큰 ID:', tokenizer.encode('칩'), '  " 칩"의 토큰 ID:', tokenizer.encode(' 칩'))

## 예제 1-2. 다음 토큰의 확률 보기

교재의 절: "다음 토큰의 확률 분포"

"웨이퍼를 개별 칩으로 자르는 공정을" 다음에 올 토큰의 확률 분포를 계산하고 상위 10개를 출력합니다. logit의 개수, 확률의 합, 1위 토큰의 확률을 봅니다.

In [ ]:
prompt = '웨이퍼를 개별 칩으로 자르는 공정을'
input_ids = tokenizer(prompt, return_tensors='pt').input_ids  # 토큰 ID의 나열. 'pt'는 PyTorch 배열로 받는다는 뜻
with torch.no_grad():                                          # 확률만 계산하고 학습에 쓰는 값은 저장하지 않습니다
    logits = model(input_ids).logits[0, -1]                    # [0, -1]: 첫 번째(유일한) 입력의 맨 끝 위치
probabilities = torch.softmax(logits, dim=-1)
print(f'logit 개수: {len(logits):,}개, 확률의 합: {probabilities.sum():.3f}')

top = torch.topk(probabilities, 10)                            # 확률이 큰 순서로 10개
for probability, token_id in zip(top.values, top.indices):
    print(f'{tokenizer.decode([token_id])!r:10} logit {logits[token_id]:6.2f}  확률 {probability:6.1%}')

## 예제 1-3. temperature가 분포를 바꾸는 모습

교재의 절: "샘플링과 temperature"

예제 1-2의 logit에 T=0.3, 1.0, 2.0을 적용해 상위 3개 토큰의 확률을 비교합니다. 이어서 같은 T로 문장을 두 번씩 생성합니다. T가 커질수록 1위의 확률과 생성한 문장이 어떻게 바뀌는지 봅니다. 시드(seed)는 난수를 만드는 시작값입니다. `top_k`, `top_p`는 이 절 뒤의 "top-k와 top-p"에서 설명합니다.

In [ ]:
for temperature in [0.3, 1.0, 2.0]:
    scaled = torch.softmax(logits / temperature, dim=-1)         # logit을 T로 나눈 뒤 softmax
    top = torch.topk(scaled, 3)
    summary = ', '.join(f'{tokenizer.decode([token_id])!r} {probability:.1%}'
                        for probability, token_id in zip(top.values, top.indices))
    print(f'T={temperature}  상위 3개: {summary}')

torch.manual_seed(0)                                              # 시드를 고정해 같은 PC에서는 같은 문장이 나오게 합니다
for temperature in [0.3, 1.0, 2.0]:
    for attempt in range(2):
        # generate()는 네 단계를 max_new_tokens번 되풀이합니다. do_sample=True는 샘플링한다는 뜻이고,
        # top_k=0, top_p=1.0, repetition_penalty=1.0은 이 모델의 기본 조정을 꺼서 temperature만의 효과를 봅니다
        output = model.generate(input_ids, do_sample=True, temperature=temperature, top_k=0, top_p=1.0,
                                repetition_penalty=1.0, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
        continuation = tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)
        print(f'\nT={temperature}: {prompt}{continuation}'.replace('\n', ' ').strip())

## 예제 1-4. 같은 질문 세 번

교재의 절: "같은 질문에 응답이 매번 다른 이유"

같은 질문을 서비스 모델에 세 번 보내 표현, 문장 순서, 덧붙인 내용이 어떻게 다른지 봅니다. `llm.ask()`는 공통 호출 함수 `llm.py`의 함수로, 질문을 API로 보내고 응답(`reply.text`)을 돌려줍니다. `model='소형'`은 모델 등급 가운데 가장 작은 모델입니다(5장 "LLM API"에서 자세히 다룹니다). 이 셀은 API를 3번 호출합니다.

In [ ]:
question = '반도체 후공정의 다이싱 공정을 두 문장으로 설명해 주세요.'
for attempt in range(1, 4):
    reply = llm.ask(question, model='소형', max_tokens=1000)
    print(f'[{attempt}회] {reply.text}\n')

## 실습 1-1. 토큰 수를 먼저 예측하기

교재의 절: "개념 확인 실습"

후공정 문장 다섯 개의 토큰 수를 셉니다. 실행하기 전에 `my_ranking`에 토큰이 많을 것 같은 순서로 이름을 적습니다. `token_count()`는 예제 1-1의 `tokenizer`로 토큰 수를 세어 돌려주도록 작성합니다. 예측과 다른 문장의 이유를 BPE로 한 문장씩 설명하면 끝납니다. 힌트: `tokenizer.encode()`가 돌려주는 목록의 길이가 토큰 수입니다.

In [ ]:
sentences = {
    '한국어': '패키지 테스트에서 불량으로 판정된 칩은 별도 트레이로 옮깁니다.',
    '영어': 'Chips that fail the package test are moved to a separate tray.',
    '숫자와 단위': '리플로 최고 온도 260°C, 유지 시간 10초, 승온 속도 3°C/s',
    '제품 코드': 'QFN-48, BGA-256, SOP-8',
    '파이썬 코드': 'for chip in wafer.chips: print(chip.id)',
}
my_ranking = ['숫자와 단위', '한국어', '제품 코드', '영어', '파이썬 코드']


def token_count(text):
    return len(tokenizer.encode(text))


counts = {name: token_count(sentence) for name, sentence in sentences.items()}
for name, sentence in sentences.items():
    print(f'{name}: {len(sentence)}자, {counts[name]}토큰')
print('\n예측한 순서:', my_ranking)
print('실제 순서:  ', sorted(counts, key=counts.get, reverse=True))
print('\n숫자와 단위:', ' | '.join(tokenizer.decode([token_id]) for token_id in tokenizer.encode(sentences['숫자와 단위'])))

## 실습 1-2. top-p 함수 작성하기

교재의 절: "개념 확인 실습"

`top_p_filter(probabilities, p)`를 작성합니다. 확률이 높은 토큰부터 차례로 더해 누적 확률이 처음으로 p 이상이 되는 토큰까지 남기고, 남은 토큰 ID 목록을 돌려줍니다. 실행하기 전에 `my_guess`에 세 경우의 남는 후보 수를 예측해 적습니다. 셋째 경우는 temperature 0.3의 분포입니다. 예제 1-2를 먼저 실행해야 합니다. 예측과 다른 경우의 이유를 분포의 형태로 설명하면 끝납니다.

In [ ]:
def top_p_filter(probabilities, p):
    """누적 확률이 처음으로 p 이상이 되는 토큰까지, 확률이 높은 순서로 토큰 ID 목록을 돌려줍니다."""
    sorted_probabilities, sorted_ids = torch.sort(probabilities, descending=True)
    cumulative = torch.cumsum(sorted_probabilities, dim=0)
    keep = int((cumulative < p).sum()) + 1
    return sorted_ids[:keep].tolist()


my_guess = {'T=1.0, p=0.5': 10, 'T=1.0, p=0.9': 50, 'T=0.3, p=0.9': 3}

for temperature, p in [(1.0, 0.5), (1.0, 0.9), (0.3, 0.9)]:
    kept = top_p_filter(torch.softmax(logits / temperature, dim=-1), p)
    guess = my_guess[f'T={temperature}, p={p}']
    print(f'T={temperature}, p={p}: 후보 {len(kept):,}개 (예측 {guess})', [tokenizer.decode([token_id]) for token_id in kept[:5]])